# 08B — Pengembangan Transfer Learning Ringkas untuk Klasifikasi Citra Awan

Notebook ini merupakan eksperimen alternatif untuk melatih model klasifikasi tujuh jenis awan pada Ground-based Cloud Dataset (GCD). Notebook dibuat terpisah sehingga tidak mengubah `08_training_model.ipynb` maupun checkpoint yang telah dihasilkan sebelumnya.

Pengembangan mengadaptasi prinsip transfer learning dari tiga referensi. Tutorial [GeeksforGeeks](https://www.geeksforgeeks.org/deep-learning/how-to-implement-transfer-learning-in-pytorch/) menjelaskan tahapan pemuatan model pretrained, penggantian classifier, pembekuan feature extractor, training classifier, dan fine-tuning menggunakan learning rate yang lebih kecil.

Penelitian [Guzel et al. (2024)](https://pmc.ncbi.nlm.nih.gov/articles/PMC10773838/) membandingkan beberapa arsitektur pretrained untuk klasifikasi jenis awan dan menggunakan pendekatan freeze-out fine-tuning. Penelitian tersebut membekukan sebagian lapisan awal agar fitur umum yang telah dipelajari model pretrained tidak langsung berubah selama proses adaptasi.

Penelitian [Kalkan et al. (2022)](https://www.sciencedirect.com/science/article/abs/pii/S0045790622004980) menunjukkan bahwa transfer learning dan freeze-out fine-tuning dapat meningkatkan hasil klasifikasi citra cuaca. Akan tetapi, penelitian tersebut menggunakan klasifikasi biner clear/cloudy, sedangkan notebook ini tetap menggunakan tujuh kelas GCD. Oleh karena itu, strategi training-nya dapat diadaptasi, tetapi nilai akurasi penelitian tidak dapat dibandingkan secara langsung.

Implementasi menggunakan `timm.create_model()` supaya pemuatan backbone pretrained dan penggantian classifier dapat dilakukan tanpa menulis ulang arsitektur ResNet secara manual.

In [ ]:
from pathlib import Path  # Mengimpor Path untuk menangani lokasi folder secara lintas sistem operasi.
from datetime import datetime  # Mengimpor datetime untuk membuat identitas waktu eksperimen.
from contextlib import nullcontext  # Mengimpor nullcontext untuk kondisi ketika mixed precision tidak digunakan.
import json  # Mengimpor json untuk membaca dan menyimpan konfigurasi eksperimen.
import random  # Mengimpor random untuk mengatur seed generator acak Python.
import re  # Mengimpor re untuk membersihkan nama model sebelum digunakan sebagai nama file.
import time  # Mengimpor time untuk menghitung durasi setiap epoch.

import albumentations as A  # Mengimpor Albumentations untuk memuat transform yang dibuat notebook 05.
import matplotlib.pyplot as plt  # Mengimpor Matplotlib untuk membuat grafik training.
import numpy as np  # Mengimpor NumPy untuk perhitungan distribusi dan bobot kelas.
import pandas as pd  # Mengimpor Pandas untuk menyimpan riwayat training sebagai tabel.
import torch  # Mengimpor PyTorch sebagai framework deep learning utama.
import torch.nn as nn  # Mengimpor modul neural network PyTorch.
import timm  # Mengimpor timm untuk memuat model pretrained secara ringkas.

from IPython.display import display  # Mengimpor display untuk menampilkan DataFrame pada notebook.
from PIL import Image  # Mengimpor Image untuk menangani citra yang dibaca ImageFolder.
from torch.optim import AdamW  # Mengimpor optimizer AdamW untuk proses optimasi model.
from torch.optim.lr_scheduler import ReduceLROnPlateau  # Mengimpor scheduler untuk menurunkan learning rate.
from torch.utils.data import DataLoader  # Mengimpor DataLoader untuk membentuk batch data.
from torchvision.datasets import ImageFolder  # Mengimpor ImageFolder untuk membaca dataset berdasarkan folder kelas.
from tqdm.auto import tqdm  # Mengimpor tqdm untuk menampilkan progress bar training.

print(f"PyTorch : {torch.__version__}")  # Menampilkan versi PyTorch yang sedang digunakan.
print(f"timm    : {timm.__version__}")  # Menampilkan versi timm yang sedang digunakan.
print(f"CUDA    : {torch.cuda.is_available()}")  # Menampilkan status ketersediaan CUDA.

## 2. Menentukan Lokasi Project

Notebook dijalankan langsung melalui VS Code/Jupyter. Oleh karena itu, lokasi project ditentukan berdasarkan working directory dan tidak menggunakan path Docker absolut `/app`.

Jika notebook dijalankan dari folder `notebooks`, maka direktori project adalah parent dari folder tersebut. Jika dijalankan dari root project, working directory langsung digunakan sebagai direktori project.

Notebook baru hanya membaca artefak dari notebook sebelumnya. Artefak tersebut tidak diubah.

In [ ]:
CURRENT_DIR = Path.cwd().resolve()  # Mengambil working directory aktif dan mengubahnya menjadi path absolut.

if CURRENT_DIR.name == "notebooks":  # Memeriksa apakah notebook dijalankan dari folder notebooks.
    PROJECT_DIR = CURRENT_DIR.parent  # Menggunakan parent folder notebooks sebagai root project.
else:  # Menangani kondisi ketika notebook dijalankan dari root project.
    PROJECT_DIR = CURRENT_DIR  # Menggunakan working directory sebagai root project.

DATASET_DIR = PROJECT_DIR / "dataset"  # Menentukan lokasi folder utama dataset.
SOURCE_DIR = DATASET_DIR / "source"  # Menentukan lokasi dataset hasil persiapan eksperimen.
PROCESSED_DIR = DATASET_DIR / "processed"  # Menentukan lokasi metadata dan konfigurasi hasil preprocessing.
MODELS_DIR = PROJECT_DIR / "models"  # Menentukan lokasi penyimpanan checkpoint model.
LOGS_DIR = PROJECT_DIR / "logs"  # Menentukan lokasi penyimpanan log eksperimen.

TRAIN_DIR = SOURCE_DIR / "train"  # Menentukan lokasi split training.
VAL_DIR = SOURCE_DIR / "val"  # Menentukan lokasi split validation.

DATALOADER_CONFIG_PATH = PROCESSED_DIR / "dataloader_config.json"  # Menentukan lokasi konfigurasi DataLoader.
MODEL_CONFIG_PATH = PROCESSED_DIR / "model_config.json"  # Menentukan lokasi konfigurasi model.
TRAIN_TRANSFORM_PATH = PROCESSED_DIR / "train_transform.json"  # Menentukan lokasi transform training.
EVAL_TRANSFORM_PATH = PROCESSED_DIR / "eval_transform.json"  # Menentukan lokasi transform evaluasi.

required_paths = [  # Membuat daftar folder dan file yang wajib tersedia.
    TRAIN_DIR,  # Memasukkan folder training ke daftar pemeriksaan.
    VAL_DIR,  # Memasukkan folder validation ke daftar pemeriksaan.
    DATALOADER_CONFIG_PATH,  # Memasukkan konfigurasi DataLoader ke daftar pemeriksaan.
    MODEL_CONFIG_PATH,  # Memasukkan konfigurasi model ke daftar pemeriksaan.
    TRAIN_TRANSFORM_PATH,  # Memasukkan transform training ke daftar pemeriksaan.
    EVAL_TRANSFORM_PATH,  # Memasukkan transform evaluasi ke daftar pemeriksaan.
]  # Menutup daftar path wajib.

missing_paths = [path for path in required_paths if not path.exists()]  # Mencari seluruh path wajib yang belum tersedia.

if missing_paths:  # Memeriksa apakah terdapat path yang belum tersedia.
    missing_text = "\n".join(f"- {path}" for path in missing_paths)  # Menyusun daftar path yang tidak ditemukan.
    raise FileNotFoundError(  # Menghentikan notebook jika tahap sebelumnya belum lengkap.
        "Prasyarat training belum tersedia:\n"  # Menjelaskan penyebab penghentian.
        f"{missing_text}\n\n"  # Menampilkan seluruh path yang belum ditemukan.
        "Jalankan notebook 05, 06, dan 07 secara berurutan."  # Memberikan petunjuk penyelesaian.
    )  # Menutup FileNotFoundError.

MODELS_DIR.mkdir(parents=True, exist_ok=True)  # Memastikan folder models tersedia tanpa menghapus isinya.
LOGS_DIR.mkdir(parents=True, exist_ok=True)  # Memastikan folder logs tersedia tanpa menghapus isinya.

path_table = pd.DataFrame({  # Membentuk tabel pemeriksaan lokasi.
    "Nama": [  # Membuat kolom nama artefak.
        "PROJECT_DIR",  # Menambahkan nama root project.
        "TRAIN_DIR",  # Menambahkan nama folder training.
        "VAL_DIR",  # Menambahkan nama folder validation.
        "MODEL_CONFIG",  # Menambahkan nama konfigurasi model.
        "DATALOADER_CONFIG",  # Menambahkan nama konfigurasi DataLoader.
    ],  # Menutup daftar nama.
    "Path": [  # Membuat kolom lokasi artefak.
        PROJECT_DIR,  # Menambahkan path root project.
        TRAIN_DIR,  # Menambahkan path training.
        VAL_DIR,  # Menambahkan path validation.
        MODEL_CONFIG_PATH,  # Menambahkan path konfigurasi model.
        DATALOADER_CONFIG_PATH,  # Menambahkan path konfigurasi DataLoader.
    ],  # Menutup daftar path.
})  # Menutup pembuatan DataFrame.

path_table["Ada"] = path_table["Path"].map(Path.exists)  # Memeriksa keberadaan setiap path.
display(path_table)  # Menampilkan tabel pemeriksaan path.

## 3. Membaca Konfigurasi Eksperimen

Konfigurasi tidak ditulis ulang agar nilai jumlah kelas, mapping label, ukuran input, batch size, dropout, dan label smoothing tetap konsisten dengan notebook sebelumnya.

Notebook mengharuskan `pretrained_loaded=True`. Jika bobot pretrained belum berhasil dimuat pada notebook 07, eksperimen dihentikan karena training tanpa bobot pretrained tidak lagi dapat disebut transfer learning.

In [ ]:
with DATALOADER_CONFIG_PATH.open("r", encoding="utf-8") as file:  # Membuka konfigurasi DataLoader dalam mode baca.
    dataloader_config = json.load(file)  # Membaca konfigurasi DataLoader menjadi dictionary Python.

with MODEL_CONFIG_PATH.open("r", encoding="utf-8") as file:  # Membuka konfigurasi model dalam mode baca.
    model_config = json.load(file)  # Membaca konfigurasi model menjadi dictionary Python.

MODEL_NAME = str(model_config["model_name"])  # Mengambil nama arsitektur model dari konfigurasi.
NUM_CLASSES = int(model_config["num_classes"])  # Mengambil jumlah kelas target.
CLASS_TO_IDX = {str(name): int(index) for name, index in model_config["class_to_idx"].items()}  # Mengambil mapping nama kelas ke indeks.
IDX_TO_CLASS = {index: name for name, index in CLASS_TO_IDX.items()}  # Membalik mapping untuk memperoleh nama kelas dari indeks.

INPUT_CHANNELS = int(model_config["input_shape"][0])  # Mengambil jumlah channel input model.
IMAGE_HEIGHT = int(model_config["input_shape"][1])  # Mengambil tinggi citra input model.
IMAGE_WIDTH = int(model_config["input_shape"][2])  # Mengambil lebar citra input model.
DROPOUT_RATE = float(model_config.get("dropout_rate", 0.0))  # Mengambil nilai dropout dengan nilai default nol.
LABEL_SMOOTHING = float(model_config.get("loss", {}).get("label_smoothing", 0.0))  # Mengambil nilai label smoothing.
PRETRAINED_AVAILABLE = bool(model_config.get("pretrained_loaded", False))  # Membaca status keberhasilan pemuatan bobot pretrained.

SEED = int(dataloader_config.get("random_seed", model_config.get("seed", 42)))  # Mengambil seed eksperimen.
BATCH_SIZE = int(dataloader_config.get("batch_size", 32))  # Mengambil batch size dari konfigurasi.
NUM_WORKERS = int(dataloader_config.get("num_workers", 0))  # Mengambil jumlah worker DataLoader.

if NUM_CLASSES != len(CLASS_TO_IDX):  # Memeriksa konsistensi jumlah kelas.
    raise ValueError("Jumlah kelas tidak konsisten dengan class_to_idx.")  # Menghentikan proses jika jumlah kelas berbeda.

if sorted(CLASS_TO_IDX.values()) != list(range(NUM_CLASSES)):  # Memeriksa apakah indeks kelas dimulai dari nol dan berurutan.
    raise ValueError("Indeks kelas harus berurutan mulai dari 0.")  # Menghentikan proses jika indeks kelas tidak valid.

if not PRETRAINED_AVAILABLE:  # Memeriksa apakah notebook 07 berhasil memuat bobot pretrained.
    raise RuntimeError(  # Menghentikan eksperimen jika bobot pretrained belum tersedia.
        "model_config.json menunjukkan pretrained_loaded=False. "  # Menjelaskan status konfigurasi.
        "Jalankan kembali notebook 07 hingga bobot pretrained berhasil dimuat."  # Memberikan tindakan perbaikan.
    )  # Menutup RuntimeError.

display(pd.DataFrame({  # Membentuk tabel ringkasan konfigurasi.
    "Parameter": [  # Membuat daftar nama parameter.
        "Model",  # Menambahkan parameter model.
        "Jumlah kelas",  # Menambahkan parameter jumlah kelas.
        "Input shape",  # Menambahkan parameter bentuk input.
        "Batch size",  # Menambahkan parameter batch size.
        "Num workers",  # Menambahkan parameter worker.
        "Dropout",  # Menambahkan parameter dropout.
        "Label smoothing",  # Menambahkan parameter label smoothing.
        "Pretrained",  # Menambahkan status pretrained.
    ],  # Menutup daftar parameter.
    "Nilai": [  # Membuat daftar nilai konfigurasi.
        MODEL_NAME,  # Menambahkan nama model.
        NUM_CLASSES,  # Menambahkan jumlah kelas.
        [INPUT_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH],  # Menambahkan bentuk input model.
        BATCH_SIZE,  # Menambahkan batch size.
        NUM_WORKERS,  # Menambahkan jumlah worker.
        DROPOUT_RATE,  # Menambahkan dropout.
        LABEL_SMOOTHING,  # Menambahkan label smoothing.
        PRETRAINED_AVAILABLE,  # Menambahkan status pretrained.
    ],  # Menutup daftar nilai.
}))  # Menutup dan menampilkan DataFrame konfigurasi.

## 4. Reproduksibilitas, Perangkat, dan Hyperparameter

Seed digunakan pada Python, NumPy, dan PyTorch untuk mengurangi variasi hasil akibat proses acak. Reproduksibilitas penuh pada GPU tidak selalu dapat dijamin karena beberapa operasi CUDA dapat memiliki implementasi nondeterministik, tetapi konfigurasi ini membuat eksperimen lebih terkendali.

Nilai hyperparameter berikut merupakan konfigurasi awal eksperimen, bukan nilai yang diklaim sebagai konfigurasi terbaik dari artikel:

- warm-up selama maksimum 5 epoch;
- fine-tuning selama maksimum 20 epoch;
- learning rate classifier `1e-3`;
- learning rate fine-tuning `1e-4`;
- early stopping setelah 5 epoch tanpa perbaikan validation loss.

Pemilihan learning rate fine-tuning yang lebih kecil bertujuan mencegah perubahan bobot pretrained secara terlalu agresif.

In [ ]:
def seed_everything(seed: int) -> None:  # Mendefinisikan fungsi untuk mengatur seluruh seed eksperimen.
    random.seed(seed)  # Mengatur seed generator acak Python.
    np.random.seed(seed)  # Mengatur seed generator acak NumPy.
    torch.manual_seed(seed)  # Mengatur seed generator acak PyTorch pada CPU.

    if torch.cuda.is_available():  # Memeriksa apakah CUDA tersedia.
        torch.cuda.manual_seed(seed)  # Mengatur seed CUDA pada GPU aktif.
        torch.cuda.manual_seed_all(seed)  # Mengatur seed CUDA pada seluruh GPU.

    if torch.backends.cudnn.is_available():  # Memeriksa apakah backend cuDNN tersedia.
        torch.backends.cudnn.deterministic = True  # Meminta cuDNN menggunakan operasi yang lebih deterministik.
        torch.backends.cudnn.benchmark = False  # Menonaktifkan pencarian algoritma tercepat yang dapat mengubah hasil.

seed_everything(SEED)  # Menjalankan pengaturan seed menggunakan nilai dari konfigurasi.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Memilih GPU jika tersedia dan CPU jika tidak tersedia.
AMP_ENABLED = DEVICE.type == "cuda"  # Mengaktifkan mixed precision hanya ketika menggunakan CUDA.
PIN_MEMORY = DEVICE.type == "cuda"  # Mengaktifkan pinned memory hanya ketika menggunakan CUDA.
NON_BLOCKING = DEVICE.type == "cuda"  # Mengaktifkan transfer non-blocking hanya ketika menggunakan CUDA.

WARMUP_EPOCHS = 5  # Menentukan jumlah maksimum epoch untuk training classifier.
FINETUNE_EPOCHS = 20  # Menentukan jumlah maksimum epoch untuk fine-tuning.
WARMUP_LEARNING_RATE = 1e-3  # Menentukan learning rate awal untuk classifier.
FINETUNE_LEARNING_RATE = 1e-4  # Menentukan learning rate lebih kecil untuk fine-tuning.
WEIGHT_DECAY = 1e-4  # Menentukan regularisasi weight decay pada optimizer.
EARLY_STOPPING_PATIENCE = 5  # Menentukan batas epoch tanpa perbaikan sebelum training dihentikan.
SCHEDULER_PATIENCE = 2  # Menentukan batas epoch sebelum learning rate diturunkan.
SCHEDULER_FACTOR = 0.5  # Menentukan faktor pengali ketika learning rate diturunkan.
MIN_LEARNING_RATE = 1e-7  # Menentukan learning rate minimum.
GRADIENT_CLIP_NORM = 1.0  # Menentukan batas maksimum norm gradient.
FREEZE_FRACTION = 1 / 3  # Menentukan sekitar sepertiga blok awal tetap dibekukan.

EXPERIMENT_TAG = "reference_tl"  # Menentukan identitas eksperimen agar tidak menimpa hasil notebook lama.
RUN_ID = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")  # Membuat identitas eksperimen berdasarkan waktu.
SAFE_MODEL_NAME = re.sub(r"[^A-Za-z0-9_.-]+", "_", MODEL_NAME)  # Membersihkan nama model agar aman digunakan sebagai nama file.

RUN_LOG_DIR = LOGS_DIR / f"{SAFE_MODEL_NAME}_{EXPERIMENT_TAG}_{RUN_ID}"  # Membuat lokasi log khusus eksperimen baru.
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)  # Membuat folder log tanpa menghapus folder yang sudah ada.

BEST_CHECKPOINT_PATH = MODELS_DIR / f"{SAFE_MODEL_NAME}_gcd_{EXPERIMENT_TAG}_best.pth"  # Menentukan checkpoint terbaik dengan nama baru.
LAST_CHECKPOINT_PATH = MODELS_DIR / f"{SAFE_MODEL_NAME}_gcd_{EXPERIMENT_TAG}_last.pth"  # Menentukan checkpoint terakhir dengan nama baru.
HISTORY_PATH = RUN_LOG_DIR / "training_history.csv"  # Menentukan lokasi riwayat training.
TRAINING_CONFIG_PATH = RUN_LOG_DIR / "training_config.json"  # Menentukan lokasi konfigurasi training.
CURVE_PATH = RUN_LOG_DIR / "learning_curves.png"  # Menentukan lokasi grafik training.

print(f"Device                    : {DEVICE}")  # Menampilkan perangkat komputasi.
print(f"Mixed precision           : {AMP_ENABLED}")  # Menampilkan status mixed precision.
print(f"Checkpoint terbaik baru   : {BEST_CHECKPOINT_PATH.relative_to(PROJECT_DIR)}")  # Menampilkan checkpoint terbaik baru.
print(f"Checkpoint terakhir baru  : {LAST_CHECKPOINT_PATH.relative_to(PROJECT_DIR)}")  # Menampilkan checkpoint terakhir baru.
print(f"Folder log baru           : {RUN_LOG_DIR.relative_to(PROJECT_DIR)}")  # Menampilkan folder log eksperimen baru.

## 5. Memuat Transform dan Dataset

Transform training dan validation tidak ditulis ulang. Notebook memuat `train_transform.json` dan `eval_transform.json` yang telah dibuat pada notebook preprocessing.

Dataset dibaca menggunakan `ImageFolder`. Nama folder kelas dan indeks kelas diperiksa terhadap `model_config.json`. Pemeriksaan ini penting karena kesalahan mapping kelas dapat menyebabkan prediksi terlihat benar secara numerik tetapi mengarah pada nama kelas yang salah.

Bobot kelas dihitung menggunakan distribusi data training:

\[
w_c = \frac{N}{K \times n_c}
\]

dengan:

- \(N\) sebagai jumlah seluruh citra training;
- \(K\) sebagai jumlah kelas;
- \(n_c\) sebagai jumlah citra pada kelas ke-\(c\).

Kelas dengan jumlah sampel lebih kecil mendapatkan bobot loss lebih besar.

In [ ]:
class AlbumentationsImageTransform:  # Mendefinisikan adapter agar transform Albumentations dapat digunakan ImageFolder.
    def __init__(self, pipeline):  # Mendefinisikan konstruktor adapter transform.
        self.pipeline = pipeline  # Menyimpan pipeline Albumentations pada objek.

    def __call__(self, image):  # Mendefinisikan proses transform ketika citra dipanggil ImageFolder.
        image_array = np.asarray(image.convert("RGB"))  # Mengubah citra PIL menjadi array RGB NumPy.
        transformed = self.pipeline(image=image_array)["image"]  # Menjalankan pipeline Albumentations dan mengambil hasil citra.

        if isinstance(transformed, np.ndarray):  # Memeriksa apakah hasil transform masih berupa array NumPy.
            transformed = np.ascontiguousarray(transformed.transpose(2, 0, 1))  # Mengubah urutan dimensi HWC menjadi CHW.
            transformed = torch.from_numpy(transformed)  # Mengubah array NumPy menjadi tensor PyTorch.

        if transformed.dtype == torch.uint8:  # Memeriksa apakah tensor masih menggunakan tipe data uint8.
            transformed = transformed.float() / 255.0  # Mengubah tensor ke float dan rentang nol sampai satu.
        else:  # Menangani tensor yang sudah memiliki tipe selain uint8.
            transformed = transformed.float()  # Memastikan tensor menggunakan tipe float32.

        return transformed  # Mengembalikan tensor citra hasil transform.

train_pipeline = A.load(str(TRAIN_TRANSFORM_PATH), data_format="json")  # Memuat pipeline augmentasi training.
eval_pipeline = A.load(str(EVAL_TRANSFORM_PATH), data_format="json")  # Memuat pipeline transform validation.

if hasattr(train_pipeline, "set_random_seed"):  # Memeriksa dukungan pengaturan seed pada pipeline training.
    train_pipeline.set_random_seed(SEED)  # Mengatur seed pipeline augmentasi training.

if hasattr(eval_pipeline, "set_random_seed"):  # Memeriksa dukungan pengaturan seed pada pipeline validation.
    eval_pipeline.set_random_seed(SEED)  # Mengatur seed pipeline validation.

train_transform = AlbumentationsImageTransform(train_pipeline)  # Membungkus transform training agar kompatibel dengan ImageFolder.
eval_transform = AlbumentationsImageTransform(eval_pipeline)  # Membungkus transform validation agar kompatibel dengan ImageFolder.

train_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transform)  # Membaca dataset training berdasarkan struktur folder kelas.
val_dataset = ImageFolder(root=VAL_DIR, transform=eval_transform)  # Membaca dataset validation berdasarkan struktur folder kelas.

if train_dataset.class_to_idx != CLASS_TO_IDX:  # Membandingkan mapping kelas training dengan konfigurasi model.
    raise ValueError(  # Menghentikan proses jika mapping kelas training berbeda.
        "Mapping kelas training berbeda dari model_config.json.\n"  # Menjelaskan sumber kesalahan.
        f"Dataset: {train_dataset.class_to_idx}\n"  # Menampilkan mapping dari dataset.
        f"Config : {CLASS_TO_IDX}"  # Menampilkan mapping dari konfigurasi.
    )  # Menutup ValueError.

if val_dataset.class_to_idx != CLASS_TO_IDX:  # Membandingkan mapping kelas validation dengan konfigurasi model.
    raise ValueError(  # Menghentikan proses jika mapping kelas validation berbeda.
        "Mapping kelas validation berbeda dari model_config.json.\n"  # Menjelaskan sumber kesalahan.
        f"Dataset: {val_dataset.class_to_idx}\n"  # Menampilkan mapping dari dataset.
        f"Config : {CLASS_TO_IDX}"  # Menampilkan mapping dari konfigurasi.
    )  # Menutup ValueError.

train_targets = np.asarray(train_dataset.targets, dtype=np.int64)  # Mengubah target training menjadi array NumPy.
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)  # Menghitung jumlah citra untuk setiap kelas.

if np.any(class_counts == 0):  # Memeriksa apakah terdapat kelas tanpa sampel training.
    raise ValueError("Terdapat kelas kosong pada dataset training.")  # Menghentikan proses jika terdapat kelas kosong.

class_weights_np = len(train_targets) / (NUM_CLASSES * class_counts.astype(np.float64))  # Menghitung bobot setiap kelas.
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)  # Mengubah bobot kelas menjadi tensor pada device aktif.

def seed_worker(worker_id: int) -> None:  # Mendefinisikan fungsi seed untuk setiap worker DataLoader.
    worker_seed = (SEED + worker_id) % (2**32)  # Membuat seed worker berdasarkan seed utama dan ID worker.
    random.seed(worker_seed)  # Mengatur seed Python pada worker.
    np.random.seed(worker_seed)  # Mengatur seed NumPy pada worker.

def create_loader(dataset, shuffle: bool) -> DataLoader:  # Mendefinisikan fungsi pembentuk DataLoader.
    generator = torch.Generator()  # Membuat generator acak khusus DataLoader.
    generator.manual_seed(SEED)  # Mengatur seed generator DataLoader.

    return DataLoader(  # Mengembalikan DataLoader yang telah dikonfigurasi.
        dataset=dataset,  # Menentukan dataset yang akan dimuat.
        batch_size=BATCH_SIZE,  # Menentukan jumlah citra dalam setiap batch.
        shuffle=shuffle,  # Mengacak data hanya jika parameter shuffle bernilai True.
        num_workers=NUM_WORKERS,  # Menentukan jumlah worker pembaca data.
        pin_memory=PIN_MEMORY,  # Mengaktifkan pinned memory ketika menggunakan CUDA.
        persistent_workers=NUM_WORKERS > 0,  # Mempertahankan worker jika jumlah worker lebih dari nol.
        drop_last=False,  # Memastikan seluruh sampel tetap digunakan.
        worker_init_fn=seed_worker,  # Mengatur seed masing-masing worker.
        generator=generator,  # Menggunakan generator yang sudah diberi seed.
    )  # Menutup pembuatan DataLoader.

train_loader = create_loader(train_dataset, shuffle=True)  # Membuat DataLoader training dengan pengacakan data.
val_loader = create_loader(val_dataset, shuffle=False)  # Membuat DataLoader validation tanpa pengacakan data.

class_distribution = pd.DataFrame({  # Membentuk tabel distribusi kelas.
    "class_id": list(range(NUM_CLASSES)),  # Membuat daftar indeks kelas.
    "class_name": [IDX_TO_CLASS[index] for index in range(NUM_CLASSES)],  # Mengambil nama setiap kelas.
    "train_count": class_counts,  # Memasukkan jumlah citra training per kelas.
    "loss_weight": class_weights_np,  # Memasukkan bobot loss setiap kelas.
})  # Menutup pembuatan DataFrame distribusi kelas.

display(class_distribution)  # Menampilkan distribusi dan bobot kelas.
print(f"Jumlah citra training   : {len(train_dataset):,}")  # Menampilkan jumlah citra training.
print(f"Jumlah citra validation : {len(val_dataset):,}")  # Menampilkan jumlah citra validation.
print(f"Jumlah batch training   : {len(train_loader):,}")  # Menampilkan jumlah batch training.
print(f"Jumlah batch validation : {len(val_loader):,}")  # Menampilkan jumlah batch validation.
print("Data test dimuat        : Tidak")  # Memastikan test set tidak digunakan.

## 6. Membangun Model Transfer Learning

`timm.create_model()` menerima nama arsitektur dari `model_config.json`. Parameter `pretrained=True` memuat bobot hasil training ImageNet, sedangkan `num_classes=NUM_CLASSES` secara otomatis mengganti classifier asli dengan classifier yang sesuai dengan tujuh kelas GCD.

Fungsi `configure_trainable_scope()` mengatur bagian model yang dapat dilatih.

Pada fase `classifier`, seluruh parameter backbone dibekukan dan hanya classifier baru yang dilatih.

Pada fase `fine_tune`, blok model yang memiliki parameter diidentifikasi melalui `named_children()`. Sekitar sepertiga blok pertama tetap dibekukan, sedangkan blok setelahnya diaktifkan kembali. Pendekatan ini merupakan implementasi ringkas dari freeze-out fine-tuning dan dapat digunakan pada ResNet18 maupun ResNet50.

In [ ]:
model = timm.create_model(  # Membuat model transfer learning menggunakan timm.
    MODEL_NAME,  # Menggunakan nama model dari model_config.json.
    pretrained=True,  # Memuat bobot pretrained ImageNet.
    num_classes=NUM_CLASSES,  # Mengganti classifier agar menghasilkan tujuh kelas.
    in_chans=INPUT_CHANNELS,  # Menyesuaikan jumlah channel input model.
    drop_rate=DROPOUT_RATE,  # Menggunakan dropout dari konfigurasi model.
)  # Menutup pembuatan model.

model = model.to(DEVICE)  # Memindahkan model ke GPU atau CPU yang aktif.

def configure_trainable_scope(scope: str) -> dict:  # Mendefinisikan fungsi untuk mengatur parameter yang dapat dilatih.
    model.requires_grad_(False)  # Membekukan seluruh parameter model terlebih dahulu.
    classifier = model.get_classifier()  # Mengambil classifier akhir model melalui antarmuka timm.

    if scope == "classifier":  # Memeriksa apakah fase hanya melatih classifier.
        classifier.requires_grad_(True)  # Mengaktifkan gradient pada classifier.
        return {  # Mengembalikan informasi blok yang dibekukan dan dilatih.
            "frozen": ["seluruh_backbone"],  # Menyatakan seluruh backbone dibekukan.
            "trainable": ["classifier"],  # Menyatakan hanya classifier yang dilatih.
        }  # Menutup dictionary informasi fase classifier.

    if scope != "fine_tune":  # Memeriksa validitas nama scope.
        raise ValueError("scope harus bernilai 'classifier' atau 'fine_tune'.")  # Menghentikan proses jika scope tidak valid.

    parameterized_blocks = [  # Membentuk daftar blok utama model yang memiliki parameter.
        (name, module)  # Menyimpan nama dan objek modul.
        for name, module in model.named_children()  # Mengiterasi blok tingkat atas model.
        if any(parameter.numel() > 0 for parameter in module.parameters())  # Memilih modul yang memiliki parameter.
    ]  # Menutup pembuatan daftar blok.

    freeze_count = max(1, int(np.ceil(len(parameterized_blocks) * FREEZE_FRACTION)))  # Menghitung jumlah blok awal yang dibekukan.
    frozen_blocks = parameterized_blocks[:freeze_count]  # Mengambil sekitar sepertiga blok pertama.
    trainable_blocks = parameterized_blocks[freeze_count:]  # Mengambil blok setelah bagian yang dibekukan.

    for _, module in trainable_blocks:  # Mengiterasi seluruh blok yang akan dilatih.
        module.requires_grad_(True)  # Mengaktifkan gradient pada blok tersebut.

    classifier.requires_grad_(True)  # Memastikan classifier selalu dapat dilatih.

    return {  # Mengembalikan informasi pembagian blok.
        "frozen": [name for name, _ in frozen_blocks],  # Mengembalikan nama blok yang dibekukan.
        "trainable": [name for name, _ in trainable_blocks],  # Mengembalikan nama blok yang dilatih.
    }  # Menutup dictionary informasi fine-tuning.

criterion = nn.CrossEntropyLoss(  # Membuat fungsi loss untuk klasifikasi multi-kelas.
    weight=class_weights,  # Menggunakan bobot kelas untuk mengurangi pengaruh ketidakseimbangan kelas.
    label_smoothing=LABEL_SMOOTHING,  # Menggunakan label smoothing dari konfigurasi.
)  # Menutup konfigurasi CrossEntropyLoss.

try:  # Mencoba menggunakan antarmuka GradScaler PyTorch terbaru.
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)  # Membuat GradScaler untuk mixed precision CUDA.
except (AttributeError, TypeError):  # Menangani versi PyTorch dengan antarmuka GradScaler lama.
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)  # Membuat GradScaler menggunakan antarmuka lama.

total_parameters = sum(parameter.numel() for parameter in model.parameters())  # Menghitung seluruh parameter model.
classifier_parameters = sum(parameter.numel() for parameter in model.get_classifier().parameters())  # Menghitung parameter classifier.

print(f"Model                  : {MODEL_NAME}")  # Menampilkan nama model.
print(f"Total parameter        : {total_parameters:,}")  # Menampilkan seluruh parameter.
print(f"Parameter classifier   : {classifier_parameters:,}")  # Menampilkan parameter classifier.
print(f"Classifier             : {model.get_classifier()}")  # Menampilkan struktur classifier.

## 7. Fungsi Satu Epoch

Fungsi `run_epoch()` digunakan untuk dua keadaan:

- jika optimizer diberikan, fungsi menjalankan training;
- jika optimizer bernilai `None`, fungsi menjalankan validation.

Penggunaan satu fungsi mencegah duplikasi loop training dan validation. Pada validation, PyTorch menonaktifkan perhitungan gradient sehingga penggunaan memori lebih rendah.

Batch Normalization yang berada pada bagian model beku dipertahankan dalam mode evaluasi. Langkah ini mencegah running mean dan running variance pada backbone beku berubah selama training classifier.

In [ ]:
def autocast_context():  # Mendefinisikan context manager untuk mixed precision.
    if AMP_ENABLED:  # Memeriksa apakah training menggunakan CUDA.
        return torch.autocast(device_type="cuda", dtype=torch.float16)  # Mengaktifkan perhitungan float16 pada GPU.
    return nullcontext()  # Menggunakan context kosong ketika training berjalan pada CPU.

def run_epoch(loader: DataLoader, optimizer=None, description: str = "Epoch") -> dict:  # Mendefinisikan fungsi training atau validation satu epoch.
    is_training = optimizer is not None  # Menentukan mode berdasarkan keberadaan optimizer.
    model.train(is_training)  # Mengaktifkan mode training atau evaluation pada model.

    if is_training:  # Menjalankan pengaturan tambahan hanya pada mode training.
        for module in model.modules():  # Mengiterasi seluruh modul pada model.
            if isinstance(module, nn.modules.batchnorm._BatchNorm):  # Memilih modul Batch Normalization.
                module_parameters = list(module.parameters())  # Mengambil parameter Batch Normalization.
                module_is_frozen = module_parameters and not any(parameter.requires_grad for parameter in module_parameters)  # Memeriksa apakah modul dibekukan.
                if module_is_frozen:  # Memeriksa hasil status pembekuan modul.
                    module.eval()  # Menjaga statistik Batch Normalization beku agar tidak berubah.

    total_loss = 0.0  # Menyiapkan penjumlahan loss seluruh sampel.
    total_correct = 0  # Menyiapkan penjumlahan prediksi benar.
    total_samples = 0  # Menyiapkan penjumlahan seluruh sampel.
    progress_bar = tqdm(loader, desc=description, leave=False)  # Membuat progress bar untuk DataLoader.

    for images, labels in progress_bar:  # Mengiterasi seluruh batch citra dan label.
        images = images.to(DEVICE, non_blocking=NON_BLOCKING)  # Memindahkan citra ke device aktif.
        labels = labels.to(DEVICE, non_blocking=NON_BLOCKING)  # Memindahkan label ke device aktif.

        if is_training:  # Memeriksa apakah fungsi sedang menjalankan training.
            optimizer.zero_grad(set_to_none=True)  # Menghapus gradient dari iterasi sebelumnya.

        with torch.set_grad_enabled(is_training):  # Mengaktifkan gradient hanya pada mode training.
            with autocast_context():  # Mengaktifkan mixed precision jika CUDA tersedia.
                logits = model(images)  # Menjalankan forward pass model.
                loss = criterion(logits, labels)  # Menghitung loss terhadap label aktual.

        if not torch.isfinite(loss):  # Memeriksa apakah loss berupa nilai finite.
            raise FloatingPointError(f"Loss tidak finite pada {description}.")  # Menghentikan proses jika loss NaN atau infinity.

        if is_training:  # Menjalankan backward pass hanya pada mode training.
            scaler.scale(loss).backward()  # Menghitung gradient dengan loss scaling.
            scaler.unscale_(optimizer)  # Mengembalikan skala gradient sebelum clipping.
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)  # Membatasi norm gradient.
            scaler.step(optimizer)  # Memperbarui parameter model melalui optimizer.
            scaler.update()  # Memperbarui faktor scaling untuk iterasi berikutnya.

        batch_size = labels.size(0)  # Mengambil jumlah sampel pada batch aktif.
        predictions = logits.argmax(dim=1)  # Mengambil kelas dengan logit tertinggi.

        total_loss += loss.detach().item() * batch_size  # Menjumlahkan loss berdasarkan jumlah sampel.
        total_correct += predictions.eq(labels).sum().item()  # Menjumlahkan prediksi yang benar.
        total_samples += batch_size  # Menjumlahkan seluruh sampel yang telah diproses.

        progress_bar.set_postfix(  # Memperbarui informasi progress bar.
            loss=f"{total_loss / total_samples:.4f}",  # Menampilkan rata-rata loss sementara.
            accuracy=f"{total_correct / total_samples:.4f}",  # Menampilkan akurasi sementara.
        )  # Menutup pembaruan progress bar.

    if total_samples == 0:  # Memeriksa apakah DataLoader menghasilkan sampel.
        raise RuntimeError("DataLoader tidak menghasilkan sampel.")  # Menghentikan proses jika DataLoader kosong.

    return {  # Mengembalikan metrik satu epoch.
        "loss": total_loss / total_samples,  # Mengembalikan rata-rata loss per sampel.
        "accuracy": total_correct / total_samples,  # Mengembalikan akurasi.
        "samples": total_samples,  # Mengembalikan jumlah sampel.
    }  # Menutup dictionary metrik.

## 8. Fungsi Training per Fase

Fungsi `fit_stage()` digunakan untuk fase warm-up dan fine-tuning. Perbedaannya hanya terdapat pada scope parameter, jumlah epoch, dan learning rate.

Model terbaik dipilih berdasarkan validation loss. Accuracy tetap dicatat, tetapi tidak digunakan sebagai satu-satunya dasar pemilihan checkpoint. Validation loss memberikan informasi mengenai kualitas probabilitas prediksi dan dapat mendeteksi penurunan generalisasi sebelum perubahan accuracy terlihat.

Scheduler `ReduceLROnPlateau` menurunkan learning rate ketika validation loss berhenti membaik. Early stopping menghentikan fase training apabila tidak terdapat perbaikan selama lima epoch berturut-turut.

In [ ]:
history = []  # Menyiapkan list untuk menyimpan riwayat seluruh epoch.
global_epoch = 0  # Menyiapkan penghitung epoch lintas fase.
best_val_loss = float("inf")  # Menetapkan validation loss terbaik awal sebagai tak terhingga.

def load_own_checkpoint(checkpoint_path: Path) -> dict:  # Mendefinisikan fungsi untuk memuat checkpoint yang dibuat notebook ini.
    try:  # Mencoba pemuatan menggunakan argumen weights_only.
        return torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)  # Memuat checkpoint lokal tepercaya secara lengkap.
    except TypeError:  # Menangani versi PyTorch yang belum mendukung weights_only.
        return torch.load(checkpoint_path, map_location=DEVICE)  # Memuat checkpoint menggunakan antarmuka lama.

def build_checkpoint(phase_name, phase_epoch, optimizer, scheduler, metrics) -> dict:  # Mendefinisikan pembentukan isi checkpoint.
    return {  # Mengembalikan dictionary checkpoint.
        "saved_at": datetime.now().astimezone().isoformat(timespec="seconds"),  # Menyimpan waktu checkpoint.
        "run_id": RUN_ID,  # Menyimpan identitas eksperimen.
        "experiment_tag": EXPERIMENT_TAG,  # Menyimpan nama eksperimen alternatif.
        "model_name": MODEL_NAME,  # Menyimpan nama arsitektur model.
        "model_state_dict": model.state_dict(),  # Menyimpan parameter model.
        "optimizer_state_dict": optimizer.state_dict(),  # Menyimpan status optimizer.
        "scheduler_state_dict": scheduler.state_dict(),  # Menyimpan status scheduler.
        "phase": phase_name,  # Menyimpan nama fase training.
        "phase_epoch": phase_epoch,  # Menyimpan epoch dalam fase aktif.
        "global_epoch": global_epoch,  # Menyimpan epoch keseluruhan.
        "metrics": metrics,  # Menyimpan metrik epoch.
        "class_to_idx": CLASS_TO_IDX,  # Menyimpan mapping kelas.
        "input_shape": [INPUT_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH],  # Menyimpan bentuk input.
        "dropout_rate": DROPOUT_RATE,  # Menyimpan nilai dropout.
        "label_smoothing": LABEL_SMOOTHING,  # Menyimpan nilai label smoothing.
        "pretrained_used": True,  # Menandai penggunaan bobot pretrained.
        "test_data_used": False,  # Menandai bahwa test set tidak digunakan.
    }  # Menutup dictionary checkpoint.

def fit_stage(scope: str, phase_name: str, epochs: int, learning_rate: float) -> None:  # Mendefinisikan training untuk satu fase.
    global global_epoch  # Mengizinkan pembaruan global_epoch dari dalam fungsi.
    global best_val_loss  # Mengizinkan pembaruan best_val_loss dari dalam fungsi.

    scope_info = configure_trainable_scope(scope)  # Mengatur blok model yang dapat dilatih.
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]  # Mengambil parameter aktif.

    optimizer = AdamW(  # Membuat optimizer khusus untuk fase aktif.
        trainable_parameters,  # Memberikan hanya parameter yang dapat dilatih.
        lr=learning_rate,  # Menetapkan learning rate fase aktif.
        weight_decay=WEIGHT_DECAY,  # Menetapkan regularisasi weight decay.
    )  # Menutup konfigurasi optimizer.

    scheduler = ReduceLROnPlateau(  # Membuat scheduler berdasarkan validation loss.
        optimizer,  # Menghubungkan scheduler dengan optimizer fase aktif.
        mode="min",  # Menyatakan bahwa nilai loss lebih kecil lebih baik.
        factor=SCHEDULER_FACTOR,  # Menetapkan faktor penurunan learning rate.
        patience=SCHEDULER_PATIENCE,  # Menetapkan toleransi sebelum learning rate diturunkan.
        min_lr=MIN_LEARNING_RATE,  # Menetapkan learning rate minimum.
    )  # Menutup konfigurasi scheduler.

    phase_best_loss = float("inf")  # Menyiapkan validation loss terbaik pada fase aktif.
    epochs_without_improvement = 0  # Menyiapkan penghitung epoch tanpa perbaikan.

    print(f"\nFase                    : {phase_name}")  # Menampilkan nama fase.
    print(f"Blok dibekukan          : {scope_info['frozen']}")  # Menampilkan blok yang dibekukan.
    print(f"Blok dilatih            : {scope_info['trainable']}")  # Menampilkan blok yang dilatih.
    print(f"Parameter dapat dilatih : {sum(parameter.numel() for parameter in trainable_parameters):,}")  # Menampilkan parameter aktif.

    for phase_epoch in range(1, epochs + 1):  # Mengiterasi epoch dalam fase aktif.
        global_epoch += 1  # Menambah penghitung epoch keseluruhan.
        epoch_start = time.perf_counter()  # Mencatat waktu awal epoch.

        train_metrics = run_epoch(  # Menjalankan satu epoch training.
            loader=train_loader,  # Menggunakan DataLoader training.
            optimizer=optimizer,  # Memberikan optimizer untuk mengaktifkan backward pass.
            description=f"{phase_name} train {phase_epoch}/{epochs}",  # Menentukan keterangan progress bar.
        )  # Menutup pemanggilan epoch training.

        val_metrics = run_epoch(  # Menjalankan satu epoch validation.
            loader=val_loader,  # Menggunakan DataLoader validation.
            optimizer=None,  # Tidak memberikan optimizer agar gradient dinonaktifkan.
            description=f"{phase_name} val {phase_epoch}/{epochs}",  # Menentukan keterangan progress bar.
        )  # Menutup pemanggilan epoch validation.

        scheduler.step(val_metrics["loss"])  # Memperbarui scheduler berdasarkan validation loss.
        current_learning_rate = float(optimizer.param_groups[0]["lr"])  # Mengambil learning rate setelah scheduler.
        elapsed_seconds = time.perf_counter() - epoch_start  # Menghitung durasi epoch.

        epoch_record = {  # Membentuk catatan metrik epoch.
            "run_id": RUN_ID,  # Menyimpan identitas eksperimen.
            "phase": phase_name,  # Menyimpan nama fase.
            "phase_epoch": phase_epoch,  # Menyimpan nomor epoch dalam fase.
            "global_epoch": global_epoch,  # Menyimpan nomor epoch keseluruhan.
            "train_loss": train_metrics["loss"],  # Menyimpan training loss.
            "train_accuracy": train_metrics["accuracy"],  # Menyimpan training accuracy.
            "val_loss": val_metrics["loss"],  # Menyimpan validation loss.
            "val_accuracy": val_metrics["accuracy"],  # Menyimpan validation accuracy.
            "learning_rate": current_learning_rate,  # Menyimpan learning rate.
            "elapsed_seconds": elapsed_seconds,  # Menyimpan durasi epoch.
        }  # Menutup catatan epoch.

        history.append(epoch_record)  # Menambahkan catatan epoch ke riwayat.
        checkpoint = build_checkpoint(phase_name, phase_epoch, optimizer, scheduler, epoch_record)  # Membentuk checkpoint epoch.
        torch.save(checkpoint, LAST_CHECKPOINT_PATH)  # Menyimpan checkpoint terakhir pada nama khusus eksperimen baru.

        improved_globally = val_metrics["loss"] < best_val_loss  # Memeriksa perbaikan terhadap seluruh fase.

        if improved_globally:  # Menangani checkpoint yang lebih baik secara global.
            best_val_loss = val_metrics["loss"]  # Memperbarui validation loss terbaik.
            torch.save(checkpoint, BEST_CHECKPOINT_PATH)  # Menyimpan checkpoint terbaik pada nama khusus eksperimen baru.

        if val_metrics["loss"] < phase_best_loss:  # Memeriksa perbaikan dalam fase aktif.
            phase_best_loss = val_metrics["loss"]  # Memperbarui loss terbaik fase.
            epochs_without_improvement = 0  # Mengatur ulang penghitung early stopping.
        else:  # Menangani epoch tanpa perbaikan.
            epochs_without_improvement += 1  # Menambah penghitung early stopping.

        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False, encoding="utf-8")  # Menyimpan riwayat sementara setiap epoch.

        best_marker = " | checkpoint terbaik" if improved_globally else ""  # Membuat penanda checkpoint terbaik.

        print(  # Menampilkan ringkasan epoch.
            f"[{phase_name}] {phase_epoch:02d}/{epochs:02d} | "  # Menampilkan fase dan nomor epoch.
            f"train_loss={train_metrics['loss']:.4f} | "  # Menampilkan training loss.
            f"train_acc={train_metrics['accuracy']:.4f} | "  # Menampilkan training accuracy.
            f"val_loss={val_metrics['loss']:.4f} | "  # Menampilkan validation loss.
            f"val_acc={val_metrics['accuracy']:.4f} | "  # Menampilkan validation accuracy.
            f"lr={current_learning_rate:.2e} | "  # Menampilkan learning rate.
            f"waktu={elapsed_seconds:.1f}s"  # Menampilkan durasi epoch.
            f"{best_marker}"  # Menampilkan penanda checkpoint terbaik jika tersedia.
        )  # Menutup perintah print.

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:  # Memeriksa kondisi early stopping.
            print(f"Early stopping pada fase {phase_name}.")  # Menampilkan informasi penghentian fase.
            break  # Menghentikan loop epoch pada fase aktif.

## 9. Menjalankan Warm-up dan Fine-tuning

Fase pertama hanya melatih classifier. Setelah fase pertama selesai, checkpoint terbaik dimuat kembali sebelum fine-tuning dimulai. Dengan demikian, fine-tuning tidak dimulai dari epoch warm-up terakhir apabila epoch terakhir tersebut memiliki validation loss yang lebih buruk.

Fase kedua mengaktifkan sekitar dua pertiga blok terakhir. Learning rate diturunkan dari `1e-3` menjadi `1e-4` agar perubahan bobot pretrained lebih terkendali.

In [ ]:
fit_stage(  # Menjalankan fase warm-up classifier.
    scope="classifier",  # Menentukan bahwa hanya classifier yang dilatih.
    phase_name="warmup_classifier",  # Menentukan nama fase pada log dan checkpoint.
    epochs=WARMUP_EPOCHS,  # Menentukan jumlah maksimum epoch warm-up.
    learning_rate=WARMUP_LEARNING_RATE,  # Menggunakan learning rate classifier.
)  # Menutup pemanggilan fase warm-up.

if not BEST_CHECKPOINT_PATH.is_file():  # Memeriksa keberadaan checkpoint terbaik setelah warm-up.
    raise FileNotFoundError("Checkpoint terbaik fase warm-up tidak ditemukan.")  # Menghentikan proses jika checkpoint tidak tersedia.

warmup_best_checkpoint = load_own_checkpoint(BEST_CHECKPOINT_PATH)  # Memuat checkpoint terbaik dari fase warm-up.
model.load_state_dict(warmup_best_checkpoint["model_state_dict"])  # Mengembalikan model ke parameter warm-up terbaik.

fit_stage(  # Menjalankan fase fine-tuning.
    scope="fine_tune",  # Mengaktifkan sekitar dua pertiga blok terakhir.
    phase_name="freeze_out_fine_tuning",  # Menentukan nama fase fine-tuning.
    epochs=FINETUNE_EPOCHS,  # Menentukan jumlah maksimum epoch fine-tuning.
    learning_rate=FINETUNE_LEARNING_RATE,  # Menggunakan learning rate yang lebih kecil.
)  # Menutup pemanggilan fase fine-tuning.

if not BEST_CHECKPOINT_PATH.is_file():  # Memastikan checkpoint terbaik keseluruhan tersedia.
    raise FileNotFoundError("Checkpoint terbaik eksperimen tidak ditemukan.")  # Menghentikan proses jika checkpoint tidak tersedia.

best_checkpoint = load_own_checkpoint(BEST_CHECKPOINT_PATH)  # Memuat checkpoint terbaik dari seluruh fase.
model.load_state_dict(best_checkpoint["model_state_dict"])  # Mengembalikan model ke parameter terbaik.
best_val_metrics = run_epoch(  # Menghitung kembali metrik validation checkpoint terbaik.
    loader=val_loader,  # Menggunakan DataLoader validation.
    optimizer=None,  # Menonaktifkan proses training.
    description="Verifikasi checkpoint terbaik",  # Menentukan keterangan progress bar.
)  # Menutup proses verifikasi.

print(f"Fase terbaik        : {best_checkpoint['phase']}")  # Menampilkan fase checkpoint terbaik.
print(f"Epoch terbaik       : {best_checkpoint['global_epoch']}")  # Menampilkan epoch checkpoint terbaik.
print(f"Validation loss     : {best_val_metrics['loss']:.6f}")  # Menampilkan validation loss terbaik.
print(f"Validation accuracy : {best_val_metrics['accuracy']:.4%}")  # Menampilkan validation accuracy terbaik.

## 10. Menyimpan Riwayat, Konfigurasi, dan Kurva Training

Artefak eksperimen menggunakan nama `reference_tl`, sehingga tidak menimpa checkpoint dari `08_training_model.ipynb`.

Artefak yang dihasilkan meliputi:

- checkpoint terbaik;
- checkpoint terakhir;
- riwayat training dalam CSV;
- konfigurasi eksperimen dalam JSON;
- grafik loss dan accuracy.

Nilai pada validation set hanya digunakan untuk memilih checkpoint. Nilai test set belum dihitung.

In [ ]:
history_df = pd.DataFrame(history)  # Mengubah riwayat training menjadi DataFrame.

if history_df.empty:  # Memeriksa apakah riwayat training berisi data.
    raise RuntimeError("Riwayat training kosong.")  # Menghentikan proses jika tidak ada epoch yang dijalankan.

history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8")  # Menyimpan riwayat training sebagai CSV.

training_config = {  # Membentuk konfigurasi lengkap eksperimen.
    "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),  # Menyimpan waktu penyelesaian eksperimen.
    "run_id": RUN_ID,  # Menyimpan identitas eksperimen.
    "experiment_tag": EXPERIMENT_TAG,  # Menyimpan penanda eksperimen alternatif.
    "notebook": "08b_training_transfer_learning_ringkas.ipynb",  # Menyimpan nama notebook yang disarankan.
    "method": "two_stage_transfer_learning_with_freeze_out",  # Menyimpan nama metode training.
    "model_name": MODEL_NAME,  # Menyimpan nama arsitektur model.
    "num_classes": NUM_CLASSES,  # Menyimpan jumlah kelas.
    "class_to_idx": CLASS_TO_IDX,  # Menyimpan mapping kelas.
    "input_shape": [INPUT_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH],  # Menyimpan bentuk input.
    "device": str(DEVICE),  # Menyimpan device training.
    "seed": SEED,  # Menyimpan seed eksperimen.
    "batch_size": BATCH_SIZE,  # Menyimpan batch size.
    "num_workers": NUM_WORKERS,  # Menyimpan jumlah worker.
    "mixed_precision": AMP_ENABLED,  # Menyimpan status mixed precision.
    "class_weights": class_weights_np.tolist(),  # Menyimpan bobot setiap kelas.
    "warmup": {  # Menyimpan konfigurasi fase warm-up.
        "epochs": WARMUP_EPOCHS,  # Menyimpan maksimum epoch warm-up.
        "learning_rate": WARMUP_LEARNING_RATE,  # Menyimpan learning rate warm-up.
        "scope": "classifier_only",  # Menyimpan bagian model yang dilatih.
    },  # Menutup konfigurasi warm-up.
    "fine_tuning": {  # Menyimpan konfigurasi fine-tuning.
        "epochs": FINETUNE_EPOCHS,  # Menyimpan maksimum epoch fine-tuning.
        "learning_rate": FINETUNE_LEARNING_RATE,  # Menyimpan learning rate fine-tuning.
        "freeze_fraction": FREEZE_FRACTION,  # Menyimpan proporsi blok awal yang dibekukan.
    },  # Menutup konfigurasi fine-tuning.
    "best_result": {  # Menyimpan hasil validation checkpoint terbaik.
        "phase": best_checkpoint["phase"],  # Menyimpan fase terbaik.
        "global_epoch": int(best_checkpoint["global_epoch"]),  # Menyimpan epoch terbaik.
        "validation_loss": float(best_val_metrics["loss"]),  # Menyimpan validation loss terbaik.
        "validation_accuracy": float(best_val_metrics["accuracy"]),  # Menyimpan validation accuracy terbaik.
    },  # Menutup hasil terbaik.
    "references": [  # Menyimpan referensi pengembangan metode.
        "https://www.geeksforgeeks.org/deep-learning/how-to-implement-transfer-learning-in-pytorch/",  # Menyimpan referensi implementasi PyTorch.
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC10773838/",  # Menyimpan referensi klasifikasi jenis awan.
        "https://doi.org/10.1016/j.compeleceng.2022.108271",  # Menyimpan referensi klasifikasi clear/cloudy.
    ],  # Menutup daftar referensi.
    "test_data_used": False,  # Menandai bahwa test set tidak digunakan.
}  # Menutup konfigurasi eksperimen.

with TRAINING_CONFIG_PATH.open("w", encoding="utf-8") as file:  # Membuka file konfigurasi dalam mode tulis.
    json.dump(training_config, file, indent=2, ensure_ascii=False)  # Menyimpan konfigurasi sebagai JSON yang mudah dibaca.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # Membuat dua area grafik dalam satu baris.

axes[0].plot(history_df["global_epoch"], history_df["train_loss"], marker="o", label="Train")  # Menggambar training loss.
axes[0].plot(history_df["global_epoch"], history_df["val_loss"], marker="o", label="Validation")  # Menggambar validation loss.
axes[0].set_title("Loss per Epoch")  # Menentukan judul grafik loss.
axes[0].set_xlabel("Global Epoch")  # Menentukan label sumbu horizontal grafik loss.
axes[0].set_ylabel("Cross-Entropy Loss")  # Menentukan label sumbu vertikal grafik loss.
axes[0].grid(alpha=0.3)  # Menambahkan garis bantu pada grafik loss.
axes[0].legend()  # Menampilkan legenda grafik loss.

axes[1].plot(history_df["global_epoch"], history_df["train_accuracy"], marker="o", label="Train")  # Menggambar training accuracy.
axes[1].plot(history_df["global_epoch"], history_df["val_accuracy"], marker="o", label="Validation")  # Menggambar validation accuracy.
axes[1].set_title("Accuracy per Epoch")  # Menentukan judul grafik accuracy.
axes[1].set_xlabel("Global Epoch")  # Menentukan label sumbu horizontal grafik accuracy.
axes[1].set_ylabel("Accuracy")  # Menentukan label sumbu vertikal grafik accuracy.
axes[1].set_ylim(0.0, 1.0)  # Membatasi rentang accuracy dari nol sampai satu.
axes[1].grid(alpha=0.3)  # Menambahkan garis bantu pada grafik accuracy.
axes[1].legend()  # Menampilkan legenda grafik accuracy.

plt.tight_layout()  # Merapikan jarak antargrafik.
fig.savefig(CURVE_PATH, dpi=150, bbox_inches="tight")  # Menyimpan grafik sebagai PNG.
plt.show()  # Menampilkan grafik pada notebook.

print(f"Riwayat training    : {HISTORY_PATH.relative_to(PROJECT_DIR)}")  # Menampilkan lokasi riwayat.
print(f"Konfigurasi training: {TRAINING_CONFIG_PATH.relative_to(PROJECT_DIR)}")  # Menampilkan lokasi konfigurasi.
print(f"Kurva training      : {CURVE_PATH.relative_to(PROJECT_DIR)}")  # Menampilkan lokasi grafik.

## 11. Pemeriksaan Akhir

Pemeriksaan akhir memastikan seluruh artefak eksperimen tersedia, nilai validation bersifat finite, rentang accuracy valid, dan checkpoint tidak menggunakan test set.

Notebook belum menyatakan model layak digunakan hanya berdasarkan validation accuracy. Model tetap harus melalui:

1. validasi lebih rinci;
2. pengujian pada test set;
3. confusion matrix;
4. precision, recall, dan macro F1-score;
5. analisis kesalahan;
6. visualisasi Grad-CAM.

In [ ]:
final_checks = {  # Membentuk daftar pemeriksaan akhir.
    "Checkpoint terbaik tersedia": BEST_CHECKPOINT_PATH.is_file(),  # Memeriksa checkpoint terbaik.
    "Checkpoint terakhir tersedia": LAST_CHECKPOINT_PATH.is_file(),  # Memeriksa checkpoint terakhir.
    "Riwayat training tersedia": HISTORY_PATH.is_file(),  # Memeriksa riwayat CSV.
    "Konfigurasi training tersedia": TRAINING_CONFIG_PATH.is_file(),  # Memeriksa konfigurasi JSON.
    "Kurva training tersedia": CURVE_PATH.is_file(),  # Memeriksa grafik training.
    "Validation loss finite": np.isfinite(best_val_metrics["loss"]),  # Memeriksa validation loss.
    "Validation accuracy valid": 0.0 <= best_val_metrics["accuracy"] <= 1.0,  # Memeriksa rentang accuracy.
    "Test set tidak digunakan": best_checkpoint.get("test_data_used") is False,  # Memeriksa status penggunaan test set.
}  # Menutup dictionary pemeriksaan.

final_check_table = pd.DataFrame({  # Membentuk tabel hasil pemeriksaan.
    "Pemeriksaan": list(final_checks.keys()),  # Memasukkan nama pemeriksaan.
    "Status": ["Berhasil" if status else "Gagal" for status in final_checks.values()],  # Mengubah Boolean menjadi status.
})  # Menutup pembuatan tabel.

display(final_check_table)  # Menampilkan hasil pemeriksaan akhir.

failed_checks = [name for name, status in final_checks.items() if not status]  # Mengambil pemeriksaan yang gagal.

if failed_checks:  # Memeriksa apakah terdapat kegagalan.
    failed_text = "\n".join(f"- {name}" for name in failed_checks)  # Menyusun daftar kegagalan.
    raise AssertionError(f"Pemeriksaan akhir gagal:\n{failed_text}")  # Menghentikan notebook jika pemeriksaan gagal.

print("=" * 72)  # Mencetak garis pembatas.
print("TRANSFER LEARNING REFERENSI SELESAI")  # Menampilkan status penyelesaian.
print("=" * 72)  # Mencetak garis pembatas.
print(f"Model               : {MODEL_NAME}")  # Menampilkan nama model.
print(f"Jumlah epoch aktual : {len(history_df)}")  # Menampilkan jumlah epoch yang dijalankan.
print(f"Fase terbaik        : {best_checkpoint['phase']}")  # Menampilkan fase terbaik.
print(f"Epoch terbaik       : {best_checkpoint['global_epoch']}")  # Menampilkan epoch terbaik.
print(f"Validation loss     : {best_val_metrics['loss']:.6f}")  # Menampilkan validation loss.
print(f"Validation accuracy : {best_val_metrics['accuracy']:.4%}")  # Menampilkan validation accuracy.
print(f"Checkpoint terbaik  : {BEST_CHECKPOINT_PATH.relative_to(PROJECT_DIR)}")  # Menampilkan lokasi checkpoint.
print("Data test digunakan : Tidak")  # Menegaskan test set belum digunakan.
print("=" * 72)  # Mencetak garis penutup.

## Kesimpulan

Notebook `08b_training_transfer_learning_ringkas.ipynb` merupakan eksperimen transfer learning alternatif yang tidak menggantikan notebook training lama.

Kode dibuat lebih ringkas melalui empat keputusan utama:

1. model pretrained dan classifier dibuat langsung melalui `timm.create_model()`;
2. training dan validation menggunakan satu fungsi `run_epoch()`;
3. warm-up dan fine-tuning menggunakan satu fungsi `fit_stage()`;
4. transform serta konfigurasi dari notebook sebelumnya digunakan kembali.

Model dilatih dalam dua fase. Fase pertama melatih classifier untuk mengenali tujuh kelas GCD. Fase kedua membekukan sekitar sepertiga blok awal dan melatih blok bagian akhir menggunakan learning rate lebih kecil.

Hasil notebook ini belum boleh dibandingkan langsung dengan nilai accuracy pada artikel karena terdapat perbedaan dataset, jumlah kelas, distribusi data, preprocessing, dan skenario evaluasi. Perbandingan yang valid harus dilakukan terhadap model lama menggunakan split train, validation, dan test yang sama.

## Tahap Selanjutnya

Setelah notebook selesai dijalankan, lakukan validasi menggunakan checkpoint:

```text
models/<nama_model>_gcd_reference_tl_best.pth